# Consolidated scientific results

## Scientific question

Which model/task combinations are complete, and how do discrimination, subject variability and prediction errors compare within each task and model?

## Rationale

This notebook renders the common HTML report and pooled out-of-fold ROC/precision-recall comparisons from saved artifacts. It never retrains models.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import IFrame, Markdown, display

from br_analysis.config import ProjectPaths
from br_analysis.registry import MODELS
from br_analysis.report import generate_html_report
from br_analysis.reporting import collect_metrics, model_summary
from br_analysis.scientific_reports import load_saved_run, plot_roc_pr_comparison
from br_analysis.tasks import TASKS

In [ ]:
TASK_ID = "c_vs_ip"
MODEL_ID = "riemann"
paths = ProjectPaths.from_env("report", create_dirs=False)
metrics = collect_metrics(paths.artifacts_dir)
summary = model_summary(metrics)
display(Markdown("## Completed model/task results"))
display(summary)
report_path = generate_html_report(paths.artifacts_dir)
display(IFrame(report_path.as_uri(), width="100%", height=900))

## Global ROC and precision-recall comparisons

Curves use pooled out-of-fold predictions and therefore require no retraining. Within-task model comparisons are the primary view; across-task curves for one model are descriptive because labels and prevalence change.

In [ ]:
task_runs = {
    model_id: load_saved_run(paths.artifacts_dir, TASK_ID, model_id)
    for model_id in MODELS
}
plot_roc_pr_comparison(task_runs, title=f"All completed models for {TASK_ID}")
plt.show()
model_runs = {
    task_id: load_saved_run(paths.artifacts_dir, task_id, MODEL_ID)
    for task_id in TASKS
}
plot_roc_pr_comparison(model_runs, title=f"{MODEL_ID} across completed tasks")
plt.show()

## Reproducible command

```powershell
br-analysis report
```

Use `br-analysis help` for the completion matrix and `br-analysis run --task all --model MODEL` or its inverse to fill missing combinations.

## Interpretation and limitations

Compare models primarily within the same task and split contract. ROC summarises ranking across thresholds; precision-recall exposes the precision cost under class imbalance. Aggregate improvements should also be supported by subject-level consistency, confusion diagnostics and a relevant simpler baseline. Missing combinations remain visibly marked rather than imputed.